# 02. EDA и ABC/XYZ-сегментация

Цель ноутбука: изучить структуру продаж, выделить товары с наибольшим вкладом в выручку и оценить стабильность спроса.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.config import PROCESSED_DATA_DIR
from src.segmentation import build_abc_xyz

In [ ]:
daily_sales = pd.read_parquet(PROCESSED_DATA_DIR / 'daily_sales.parquet')
daily_sales['date'] = pd.to_datetime(daily_sales['date'])
daily_sales.head()

In [ ]:
daily_total = daily_sales.groupby('date', as_index=False).agg(sales=('sales', 'sum'), revenue=('revenue', 'sum'))
daily_total.tail()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
sns.lineplot(data=daily_total, x='date', y='revenue', ax=ax)
ax.set_title('Дневная выручка')
ax.set_xlabel('Дата')
ax.set_ylabel('Выручка')
plt.tight_layout()

In [ ]:
abc_xyz = build_abc_xyz(daily_sales, ['stock_code', 'country'])
abc_xyz.to_parquet(PROCESSED_DATA_DIR / 'abc_xyz.parquet', index=False)
abc_xyz.head()

In [ ]:
segment_matrix = abc_xyz.groupby(['abc_segment', 'xyz_segment'], as_index=False).agg(
    sku_count=('stock_code', 'count'),
    revenue=('revenue', 'sum'),
)
segment_matrix

## Интерпретация после запуска

- товары группы A дают `[A]` выручки;
- товары X имеют наиболее стабильный спрос;
- товары Z требуют осторожного расчета запасов из-за высокой изменчивости.